# Table of Contents
## 1. Load 'clean.csv' dataset in dataframe as 'clean'
## 2. Creates 'day_direction_16:15' column that determines movement of price for the day
## 3. Creates 'day_direction_reg_16:00' column that determines movement of price for the day
## 4. Creates 'philo_result' that determines if the strategy was correct or incorrect
## 5. Creates 'money_made' that determines if the contract_location was crossed (profit realized)
## 6. Creates 'diff_open_close' column that shows absolute value difference on the open and closing price of the market
## 7. Creates 'rounded_diff_open_close' column that rounds the 'diff_open_close' value to the nearest integer/whole number
## 8. Creates columns 'open_close_diff_sdv_interval' and 'open_close_diff_nearest_sdv' categorizing the open-close difference
## 9. Creates 'diff_open_conLoc' column that shows absolute value difference on the open of market and 'contract_location' column
## 10. Creates columns 'open_conLoc_diff_sdv_interval' and 'open_conLoc_diff_nearest_sdv' categorizing the open-contract location difference
## 11. Creates columns 'dow_win_streak' and 'dow_loss_streak' that records the win/loss streaks for the 'time' and 'day_of_week' (Mon-Fri)
## 12. Creates columns 'overall_win_streak' and 'overall_loss_streak' that records the win/loss streaks for the 'time'
## 13. Fetches VIX data via yfinance and adds 'vix_close', 'vix_category' (Low/Moderate/Elevated/High), and 'vix_favorable' (bool: True when VIX 15-30, mean-reversion sweet spot)

## 1.

In [1]:
import pandas as pd

clean = pd.read_csv("clean.csv")

In [2]:
clean.head()

,date,candle_open_price,candle_high_price,candle_low_price,candle_close_price,volume,average,bar_count,time,confirm_time,...,day,day_of_week,market_open_09:30,initial_direction,above,below,market_open,contract_location,market_close_16:15,market_reg_close_16:00
0,2024-03-11,5182.00,5182.50,5178.25,5179.25,-1.0,-1.0,-1.0,09:31:00,09:31:00,...,11,Monday,5177.75,above,5181.0,5169.0,5177.75,5169.0,5188.0,5185.75
1,2024-03-11,5179.25,5180.25,5175.50,5179.00,-1.0,-1.0,-1.0,09:32:00,09:32:00,...,11,Monday,5177.75,above,5181.0,5169.0,5177.75,5169.0,5188.0,5185.75
2,2024-03-11,5179.00,5179.50,5174.50,5176.00,-1.0,-1.0,-1.0,09:33:00,09:33:00,...,11,Monday,5177.75,above,5181.0,5169.0,5177.75,5169.0,5188.0,5185.75
3,2024-03-11,5176.00,5179.00,5174.50,5175.25,-1.0,-1.0,-1.0,09:34:00,09:34:00,...,11,Monday,5177.75,below,5181.0,5169.0,5177.75,5181.0,5188.0,5185.75
4,2024-03-11,5175.25,5179.25,5174.00,5177.75,-1.0,-1.0,-1.0,09:35:00,09:35:00,...,11,Monday,5177.75,below,5181.0,5169.0,5177.75,5181.0,5188.0,5185.75


## 2.

In [3]:
import numpy as np

# Vectorized day direction vs 16:15 close
# 'down' = market open was ABOVE the close (price fell during day) -> strategy: opened above, expect down -> correct
# 'up'   = market open was BELOW the close (price rose during day) -> strategy: opened below, expect up  -> correct
clean['day_direction_16:15'] = np.where(
    clean['market_open_09:30'] > clean['market_close_16:15'], 'down',
    np.where(clean['market_open_09:30'] < clean['market_close_16:15'], 'up', 'equal')
)
clean.head()


,date,candle_open_price,candle_high_price,candle_low_price,candle_close_price,volume,average,bar_count,time,confirm_time,...,day_of_week,market_open_09:30,initial_direction,above,below,market_open,contract_location,market_close_16:15,market_reg_close_16:00,day_direction_16:15
0,2024-03-11,5182.00,5182.50,5178.25,5179.25,-1.0,-1.0,-1.0,09:31:00,09:31:00,...,Monday,5177.75,above,5181.0,5169.0,5177.75,5169.0,5188.0,5185.75,up
1,2024-03-11,5179.25,5180.25,5175.50,5179.00,-1.0,-1.0,-1.0,09:32:00,09:32:00,...,Monday,5177.75,above,5181.0,5169.0,5177.75,5169.0,5188.0,5185.75,up
2,2024-03-11,5179.00,5179.50,5174.50,5176.00,-1.0,-1.0,-1.0,09:33:00,09:33:00,...,Monday,5177.75,above,5181.0,5169.0,5177.75,5169.0,5188.0,5185.75,up
3,2024-03-11,5176.00,5179.00,5174.50,5175.25,-1.0,-1.0,-1.0,09:34:00,09:34:00,...,Monday,5177.75,below,5181.0,5169.0,5177.75,5181.0,5188.0,5185.75,up
4,2024-03-11,5175.25,5179.25,5174.00,5177.75,-1.0,-1.0,-1.0,09:35:00,09:35:00,...,Monday,5177.75,below,5181.0,5169.0,5177.75,5181.0,5188.0,5185.75,up


## 3.

In [4]:
import numpy as np

# Vectorized day direction vs regular session close at 16:00
clean['day_direction_reg_16:00'] = np.where(
    clean['market_open_09:30'] > clean['market_reg_close_16:00'], 'down',
    np.where(clean['market_open_09:30'] < clean['market_reg_close_16:00'], 'up', 'equal')
)
clean.head()


,date,candle_open_price,candle_high_price,candle_low_price,candle_close_price,volume,average,bar_count,time,confirm_time,...,market_open_09:30,initial_direction,above,below,market_open,contract_location,market_close_16:15,market_reg_close_16:00,day_direction_16:15,day_direction_reg_16:00
0,2024-03-11,5182.00,5182.50,5178.25,5179.25,-1.0,-1.0,-1.0,09:31:00,09:31:00,...,5177.75,above,5181.0,5169.0,5177.75,5169.0,5188.0,5185.75,up,up
1,2024-03-11,5179.25,5180.25,5175.50,5179.00,-1.0,-1.0,-1.0,09:32:00,09:32:00,...,5177.75,above,5181.0,5169.0,5177.75,5169.0,5188.0,5185.75,up,up
2,2024-03-11,5179.00,5179.50,5174.50,5176.00,-1.0,-1.0,-1.0,09:33:00,09:33:00,...,5177.75,above,5181.0,5169.0,5177.75,5169.0,5188.0,5185.75,up,up
3,2024-03-11,5176.00,5179.00,5174.50,5175.25,-1.0,-1.0,-1.0,09:34:00,09:34:00,...,5177.75,below,5181.0,5169.0,5177.75,5181.0,5188.0,5185.75,up,up
4,2024-03-11,5175.25,5179.25,5174.00,5177.75,-1.0,-1.0,-1.0,09:35:00,09:35:00,...,5177.75,below,5181.0,5169.0,5177.75,5181.0,5188.0,5185.75,up,up


## 4.

In [5]:
import numpy as np

# Was the strategy prediction correct?
# Correct: opened ABOVE and day closed DOWN (mean-reversion happened)
#       or opened BELOW and day closed UP   (mean-reversion happened)
correct_mask = (
    ((clean['initial_direction'] == 'above') & (clean['day_direction_16:15'] == 'down')) |
    ((clean['initial_direction'] == 'below') & (clean['day_direction_16:15'] == 'up'))
)
clean['philo_result'] = np.where(correct_mask, 'correct', 'incorrect')
clean.head()


,date,candle_open_price,candle_high_price,candle_low_price,candle_close_price,volume,average,bar_count,time,confirm_time,...,initial_direction,above,below,market_open,contract_location,market_close_16:15,market_reg_close_16:00,day_direction_16:15,day_direction_reg_16:00,philo_result
0,2024-03-11,5182.00,5182.50,5178.25,5179.25,-1.0,-1.0,-1.0,09:31:00,09:31:00,...,above,5181.0,5169.0,5177.75,5169.0,5188.0,5185.75,up,up,incorrect
1,2024-03-11,5179.25,5180.25,5175.50,5179.00,-1.0,-1.0,-1.0,09:32:00,09:32:00,...,above,5181.0,5169.0,5177.75,5169.0,5188.0,5185.75,up,up,incorrect
2,2024-03-11,5179.00,5179.50,5174.50,5176.00,-1.0,-1.0,-1.0,09:33:00,09:33:00,...,above,5181.0,5169.0,5177.75,5169.0,5188.0,5185.75,up,up,incorrect
3,2024-03-11,5176.00,5179.00,5174.50,5175.25,-1.0,-1.0,-1.0,09:34:00,09:34:00,...,below,5181.0,5169.0,5177.75,5181.0,5188.0,5185.75,up,up,correct
4,2024-03-11,5175.25,5179.25,5174.00,5177.75,-1.0,-1.0,-1.0,09:35:00,09:35:00,...,below,5181.0,5169.0,5177.75,5181.0,5188.0,5185.75,up,up,correct


## 5.

In [6]:
import numpy as np

# Validate first -- NaN contract_location silently produces 'no' for money_made
nan_count = clean['contract_location'].isna().sum()
if nan_count > 0:
    print(f"WARNING: {nan_count} rows have NaN contract_location -- money_made will be 'no' for these rows.")
    print("  Cause: dates beyond fixedConLocUpTo03-5-24.xlsx coverage. Add those dates to the Excel file.")
else:
    print("contract_location is fully populated -- no NaN rows.")

# Did price actually cross the contract strike by end of day?
# Opened ABOVE -> expecting DOWN reversal -> win if 16:15 close < contract_location (BELOW strike)
# Opened BELOW -> expecting UP reversal   -> win if 16:15 close > contract_location (ABOVE strike)
money_mask = (
    ((clean['initial_direction'] == 'above') & (clean['market_close_16:15'] < clean['contract_location'])) |
    ((clean['initial_direction'] == 'below') & (clean['market_close_16:15'] > clean['contract_location']))
)
clean['money_made'] = np.where(money_mask, 'yes', 'no')
clean.head()


  Cause: dates beyond fixedConLocUpTo03-5-24.xlsx coverage. Add those dates to the Excel file.


,date,candle_open_price,candle_high_price,candle_low_price,candle_close_price,volume,average,bar_count,time,confirm_time,...,above,below,market_open,contract_location,market_close_16:15,market_reg_close_16:00,day_direction_16:15,day_direction_reg_16:00,philo_result,money_made
0,2024-03-11,5182.00,5182.50,5178.25,5179.25,-1.0,-1.0,-1.0,09:31:00,09:31:00,...,5181.0,5169.0,5177.75,5169.0,5188.0,5185.75,up,up,incorrect,no
1,2024-03-11,5179.25,5180.25,5175.50,5179.00,-1.0,-1.0,-1.0,09:32:00,09:32:00,...,5181.0,5169.0,5177.75,5169.0,5188.0,5185.75,up,up,incorrect,no
2,2024-03-11,5179.00,5179.50,5174.50,5176.00,-1.0,-1.0,-1.0,09:33:00,09:33:00,...,5181.0,5169.0,5177.75,5169.0,5188.0,5185.75,up,up,incorrect,no
3,2024-03-11,5176.00,5179.00,5174.50,5175.25,-1.0,-1.0,-1.0,09:34:00,09:34:00,...,5181.0,5169.0,5177.75,5181.0,5188.0,5185.75,up,up,correct,yes
4,2024-03-11,5175.25,5179.25,5174.00,5177.75,-1.0,-1.0,-1.0,09:35:00,09:35:00,...,5181.0,5169.0,5177.75,5181.0,5188.0,5185.75,up,up,correct,yes


## 6.

In [7]:
# Vectorized absolute difference between market open and 16:15 close
# Shows how many points price moved during the day -- used for SDV volatility categorization
clean['diff_open_close'] = (clean['market_open_09:30'] - clean['market_close_16:15']).abs()
print(clean.head(75))


          date  candle_open_price  candle_high_price  candle_low_price  \
0   2024-03-11            5182.00            5182.50           5178.25   
1   2024-03-11            5179.25            5180.25           5175.50   
2   2024-03-11            5179.00            5179.50           5174.50   
3   2024-03-11            5176.00            5179.00           5174.50   
4   2024-03-11            5175.25            5179.25           5174.00   
..         ...                ...                ...               ...   
70  2024-03-12            5191.50            5197.75           5191.25   
71  2024-03-12            5200.75            5201.50           5199.50   
72  2024-03-12            5202.25            5204.00           5201.50   
73  2024-03-12            5209.00            5209.00           5206.25   
74  2024-03-12            5207.00            5211.25           5206.00   

    candle_close_price  volume  average  bar_count      time confirm_time  \
0              5179.25    -1.0    

## 7.

In [8]:
import numpy as np

clean['rounded_diff_open_close']= np.ceil(clean['diff_open_close'])

clean.head(75)

,date,candle_open_price,candle_high_price,candle_low_price,candle_close_price,volume,average,bar_count,time,confirm_time,...,market_open,contract_location,market_close_16:15,market_reg_close_16:00,day_direction_16:15,day_direction_reg_16:00,philo_result,money_made,diff_open_close,rounded_diff_open_close
0,2024-03-11,5182.00,5182.50,5178.25,5179.25,-1.0,-1.0,-1.0,09:31:00,09:31:00,...,5177.75,5169.0,5188.00,5185.75,up,up,incorrect,no,10.25,11.0
1,2024-03-11,5179.25,5180.25,5175.50,5179.00,-1.0,-1.0,-1.0,09:32:00,09:32:00,...,5177.75,5169.0,5188.00,5185.75,up,up,incorrect,no,10.25,11.0
2,2024-03-11,5179.00,5179.50,5174.50,5176.00,-1.0,-1.0,-1.0,09:33:00,09:33:00,...,5177.75,5169.0,5188.00,5185.75,up,up,incorrect,no,10.25,11.0
3,2024-03-11,5176.00,5179.00,5174.50,5175.25,-1.0,-1.0,-1.0,09:34:00,09:34:00,...,5177.75,5181.0,5188.00,5185.75,up,up,correct,yes,10.25,11.0
4,2024-03-11,5175.25,5179.25,5174.00,5177.75,-1.0,-1.0,-1.0,09:35:00,09:35:00,...,5177.75,5181.0,5188.00,5185.75,up,up,correct,yes,10.25,11.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,2024-03-12,5191.50,5197.75,5191.25,5197.75,-1.0,-1.0,-1.0,09:55:00,09:55:00,...,5206.25,5215.0,5241.75,5241.50,up,up,correct,yes,35.50,36.0
71,2024-03-12,5200.75,5201.50,5199.50,5201.00,-1.0,-1.0,-1.0,09:57:00,09:57:00,...,5206.25,5215.0,5241.75,5241.50,up,up,correct,yes,35.50,36.0
72,2024-03-12,5202.25,5204.00,5201.50,5202.75,-1.0,-1.0,-1.0,10:04:00,10:04:00,...,5206.25,5215.0,5241.75,5241.50,up,up,correct,yes,35.50,36.0
73,2024-03-12,5209.00,5209.00,5206.25,5207.00,-1.0,-1.0,-1.0,10:13:00,10:13:00,...,5206.25,5203.0,5241.75,5241.50,up,up,incorrect,no,35.50,36.0


## 8.

In [9]:
import pandas as pd

def categorize_open_close_diff(df, column_name='rounded_diff_open_close'):
    """
    Categorizes the open-close difference into standard deviation intervals.

    IMPORTANT: Bins are computed from THIS dataset's own mean and std.
    They are NOT fixed values -- they shift as more data is added over time.
    Categories are relative to the current distribution, so 'High' today
    may represent a different absolute point value after the dataset grows.

    Labels and their meaning:
      'Extremely Low'  = more than 2 std below mean (unusually small daily move)
      'Low'            = 1 to 2 std below mean
      'Slightly Low'   = 0 to 1 std below mean
      'Slightly High'  = 0 to 1 std above mean
      'High'           = 1 to 2 std above mean
      'Extremely High' = more than 2 std above mean (unusually large daily move)
    """
    mean_val = df[column_name].mean()
    std_dev = df[column_name].std()

    bins = [-float('inf'), mean_val - 2*std_dev, mean_val - std_dev, mean_val,
            mean_val + std_dev, mean_val + 2*std_dev, float('inf')]
    labels = ['Extremely Low', 'Low', 'Slightly Low', 'Slightly High', 'High', 'Extremely High']

    df['open_close_diff_sdv_interval'] = pd.cut(df[column_name], bins=bins, labels=labels)
    df['open_close_diff_nearest_sdv'] = ((df[column_name] - mean_val) / std_dev).round()
    return df

clean = categorize_open_close_diff(clean, column_name='rounded_diff_open_close')
clean.head(75)


,date,candle_open_price,candle_high_price,candle_low_price,candle_close_price,volume,average,bar_count,time,confirm_time,...,market_close_16:15,market_reg_close_16:00,day_direction_16:15,day_direction_reg_16:00,philo_result,money_made,diff_open_close,rounded_diff_open_close,open_close_diff_sdv_interval,open_close_diff_nearest_sdv
0,2024-03-11,5182.00,5182.50,5178.25,5179.25,-1.0,-1.0,-1.0,09:31:00,09:31:00,...,5188.00,5185.75,up,up,incorrect,no,10.25,11.0,Slightly Low,-1.0
1,2024-03-11,5179.25,5180.25,5175.50,5179.00,-1.0,-1.0,-1.0,09:32:00,09:32:00,...,5188.00,5185.75,up,up,incorrect,no,10.25,11.0,Slightly Low,-1.0
2,2024-03-11,5179.00,5179.50,5174.50,5176.00,-1.0,-1.0,-1.0,09:33:00,09:33:00,...,5188.00,5185.75,up,up,incorrect,no,10.25,11.0,Slightly Low,-1.0
3,2024-03-11,5176.00,5179.00,5174.50,5175.25,-1.0,-1.0,-1.0,09:34:00,09:34:00,...,5188.00,5185.75,up,up,correct,yes,10.25,11.0,Slightly Low,-1.0
4,2024-03-11,5175.25,5179.25,5174.00,5177.75,-1.0,-1.0,-1.0,09:35:00,09:35:00,...,5188.00,5185.75,up,up,correct,yes,10.25,11.0,Slightly Low,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,2024-03-12,5191.50,5197.75,5191.25,5197.75,-1.0,-1.0,-1.0,09:55:00,09:55:00,...,5241.75,5241.50,up,up,correct,yes,35.50,36.0,Slightly High,0.0
71,2024-03-12,5200.75,5201.50,5199.50,5201.00,-1.0,-1.0,-1.0,09:57:00,09:57:00,...,5241.75,5241.50,up,up,correct,yes,35.50,36.0,Slightly High,0.0
72,2024-03-12,5202.25,5204.00,5201.50,5202.75,-1.0,-1.0,-1.0,10:04:00,10:04:00,...,5241.75,5241.50,up,up,correct,yes,35.50,36.0,Slightly High,0.0
73,2024-03-12,5209.00,5209.00,5206.25,5207.00,-1.0,-1.0,-1.0,10:13:00,10:13:00,...,5241.75,5241.50,up,up,incorrect,no,35.50,36.0,Slightly High,0.0


## 9.

In [10]:
# Vectorized absolute distance between market open and contract strike
# Shows how far the target contract is from where the market opened
clean['diff_open_conLoc'] = (clean['market_open_09:30'] - clean['contract_location']).abs()
print(clean.head(75))

clean.to_csv('clean.csv', index=False)


          date  candle_open_price  candle_high_price  candle_low_price  \
0   2024-03-11            5182.00            5182.50           5178.25   
1   2024-03-11            5179.25            5180.25           5175.50   
2   2024-03-11            5179.00            5179.50           5174.50   
3   2024-03-11            5176.00            5179.00           5174.50   
4   2024-03-11            5175.25            5179.25           5174.00   
..         ...                ...                ...               ...   
70  2024-03-12            5191.50            5197.75           5191.25   
71  2024-03-12            5200.75            5201.50           5199.50   
72  2024-03-12            5202.25            5204.00           5201.50   
73  2024-03-12            5209.00            5209.00           5206.25   
74  2024-03-12            5207.00            5211.25           5206.00   

    candle_close_price  volume  average  bar_count      time confirm_time  \
0              5179.25    -1.0    

## 10.

In [11]:
import pandas as pd

def categorize_open_conLoc_diff(df, column_name='diff_open_conLoc'):
    """
    Categorizes the open-to-contract-location distance into standard deviation intervals.

    IMPORTANT: Bins are computed from THIS dataset's own mean and std -- not fixed values.
    They shift as more data is added. Categories are relative to the current distribution.

    Labels:
      'Extremely Low'  = contract strike very close to market open (tight target)
      'Extremely High' = contract strike far from market open (wide target)
    """
    mean_val = df[column_name].mean()
    std_dev = df[column_name].std()

    bins = [-float('inf'), mean_val - 2*std_dev, mean_val - std_dev, mean_val,
            mean_val + std_dev, mean_val + 2*std_dev, float('inf')]
    labels = ['Extremely Low', 'Low', 'Slightly Low', 'Slightly High', 'High', 'Extremely High']

    df['open_conLoc_diff_sdv_interval'] = pd.cut(df[column_name], bins=bins, labels=labels)
    df['open_conLoc_diff_nearest_sdv'] = ((df[column_name] - mean_val) / std_dev).round()
    return df

# BUG FIX: column_name previously had an extra quote: "'diff_open_conLoc" (caused KeyError)
clean = categorize_open_conLoc_diff(clean, column_name='diff_open_conLoc')
clean.head(75)


,date,candle_open_price,candle_high_price,candle_low_price,candle_close_price,volume,average,bar_count,time,confirm_time,...,day_direction_reg_16:00,philo_result,money_made,diff_open_close,rounded_diff_open_close,open_close_diff_sdv_interval,open_close_diff_nearest_sdv,diff_open_conLoc,open_conLoc_diff_sdv_interval,open_conLoc_diff_nearest_sdv
0,2024-03-11,5182.00,5182.50,5178.25,5179.25,-1.0,-1.0,-1.0,09:31:00,09:31:00,...,up,incorrect,no,10.25,11.0,Slightly Low,-1.0,8.75,Slightly High,0.0
1,2024-03-11,5179.25,5180.25,5175.50,5179.00,-1.0,-1.0,-1.0,09:32:00,09:32:00,...,up,incorrect,no,10.25,11.0,Slightly Low,-1.0,8.75,Slightly High,0.0
2,2024-03-11,5179.00,5179.50,5174.50,5176.00,-1.0,-1.0,-1.0,09:33:00,09:33:00,...,up,incorrect,no,10.25,11.0,Slightly Low,-1.0,8.75,Slightly High,0.0
3,2024-03-11,5176.00,5179.00,5174.50,5175.25,-1.0,-1.0,-1.0,09:34:00,09:34:00,...,up,correct,yes,10.25,11.0,Slightly Low,-1.0,3.25,Slightly Low,-1.0
4,2024-03-11,5175.25,5179.25,5174.00,5177.75,-1.0,-1.0,-1.0,09:35:00,09:35:00,...,up,correct,yes,10.25,11.0,Slightly Low,-1.0,3.25,Slightly Low,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,2024-03-12,5191.50,5197.75,5191.25,5197.75,-1.0,-1.0,-1.0,09:55:00,09:55:00,...,up,correct,yes,35.50,36.0,Slightly High,0.0,8.75,Slightly High,0.0
71,2024-03-12,5200.75,5201.50,5199.50,5201.00,-1.0,-1.0,-1.0,09:57:00,09:57:00,...,up,correct,yes,35.50,36.0,Slightly High,0.0,8.75,Slightly High,0.0
72,2024-03-12,5202.25,5204.00,5201.50,5202.75,-1.0,-1.0,-1.0,10:04:00,10:04:00,...,up,correct,yes,35.50,36.0,Slightly High,0.0,8.75,Slightly High,0.0
73,2024-03-12,5209.00,5209.00,5206.25,5207.00,-1.0,-1.0,-1.0,10:13:00,10:13:00,...,up,incorrect,no,35.50,36.0,Slightly High,0.0,3.25,Slightly Low,-1.0


## 11.

In [12]:
import numpy as np

def streak_within_group(group, target_val):
    """
    Count consecutive occurrences of target_val in chronological order.

    How the cumsum trick works:
    1. Create a True/False mask where money_made matches target_val
    2. Take cumsum of NOT-mask: each time the streak breaks, the ID increments
       -- this labels each run of consecutive matching values with a unique ID
    3. Within each ID-group, cumcount gives the position = streak length
    """
    mask = group['money_made'] == target_val
    group_ids = (~mask).cumsum()
    return (mask * (mask.groupby(group_ids).cumcount() + 1)).astype(int)

# Sort chronologically before computing streaks
clean = clean.sort_values(by=['date', 'time']).reset_index(drop=True)

# DOW streaks: consecutive wins/losses for each (day_of_week + time) combination
# e.g., all Mondays at 09:31 share one streak counter, all Tuesdays at 09:31 another
clean['dow_win_streak'] = (
    clean.groupby(['day_of_week', 'time'], group_keys=False)
    .apply(lambda g: streak_within_group(g, 'yes'))
)
clean['dow_loss_streak'] = (
    clean.groupby(['day_of_week', 'time'], group_keys=False)
    .apply(lambda g: streak_within_group(g, 'no'))
)


In [13]:
clean.head()

,date,candle_open_price,candle_high_price,candle_low_price,candle_close_price,volume,average,bar_count,time,confirm_time,...,money_made,diff_open_close,rounded_diff_open_close,open_close_diff_sdv_interval,open_close_diff_nearest_sdv,diff_open_conLoc,open_conLoc_diff_sdv_interval,open_conLoc_diff_nearest_sdv,dow_win_streak,dow_loss_streak
0,2024-03-11,5182.00,5182.50,5178.25,5179.25,-1.0,-1.0,-1.0,09:31:00,09:31:00,...,no,10.25,11.0,Slightly Low,-1.0,8.75,Slightly High,0.0,0,1
1,2024-03-11,5179.25,5180.25,5175.50,5179.00,-1.0,-1.0,-1.0,09:32:00,09:32:00,...,no,10.25,11.0,Slightly Low,-1.0,8.75,Slightly High,0.0,0,1
2,2024-03-11,5179.00,5179.50,5174.50,5176.00,-1.0,-1.0,-1.0,09:33:00,09:33:00,...,no,10.25,11.0,Slightly Low,-1.0,8.75,Slightly High,0.0,0,1
3,2024-03-11,5176.00,5179.00,5174.50,5175.25,-1.0,-1.0,-1.0,09:34:00,09:34:00,...,yes,10.25,11.0,Slightly Low,-1.0,3.25,Slightly Low,-1.0,1,0
4,2024-03-11,5175.25,5179.25,5174.00,5177.75,-1.0,-1.0,-1.0,09:35:00,09:35:00,...,yes,10.25,11.0,Slightly Low,-1.0,3.25,Slightly Low,-1.0,1,0


## 12.

In [14]:
# Overall streaks: consecutive wins/losses for each time slot across ALL days
# e.g., every day at 09:31 (Mon through Fri) shares one counter, regardless of day of week
# streak_within_group is defined in the cell above
clean["overall_win_streak"] = (
    clean.groupby("time", group_keys=False)
    .apply(lambda g: streak_within_group(g, "yes"))
)
clean["overall_loss_streak"] = (
    clean.groupby("time", group_keys=False)
    .apply(lambda g: streak_within_group(g, "no"))
)
clean.head()


,date,candle_open_price,candle_high_price,candle_low_price,candle_close_price,volume,average,bar_count,time,confirm_time,...,rounded_diff_open_close,open_close_diff_sdv_interval,open_close_diff_nearest_sdv,diff_open_conLoc,open_conLoc_diff_sdv_interval,open_conLoc_diff_nearest_sdv,dow_win_streak,dow_loss_streak,overall_win_streak,overall_loss_streak
0,2024-03-11,5182.00,5182.50,5178.25,5179.25,-1.0,-1.0,-1.0,09:31:00,09:31:00,...,11.0,Slightly Low,-1.0,8.75,Slightly High,0.0,0,1,0,1
1,2024-03-11,5179.25,5180.25,5175.50,5179.00,-1.0,-1.0,-1.0,09:32:00,09:32:00,...,11.0,Slightly Low,-1.0,8.75,Slightly High,0.0,0,1,0,1
2,2024-03-11,5179.00,5179.50,5174.50,5176.00,-1.0,-1.0,-1.0,09:33:00,09:33:00,...,11.0,Slightly Low,-1.0,8.75,Slightly High,0.0,0,1,0,1
3,2024-03-11,5176.00,5179.00,5174.50,5175.25,-1.0,-1.0,-1.0,09:34:00,09:34:00,...,11.0,Slightly Low,-1.0,3.25,Slightly Low,-1.0,1,0,1,0
4,2024-03-11,5175.25,5179.25,5174.00,5177.75,-1.0,-1.0,-1.0,09:35:00,09:35:00,...,11.0,Slightly Low,-1.0,3.25,Slightly Low,-1.0,1,0,1,0


## 13.

In [15]:
import yfinance as yf
import pandas as pd

# --- VIX Integration ---
# VIX measures expected S&P 500 volatility over the next 30 days.
# For mean-reversion: moderate VIX (15-30) is the sweet spot --
# enough noise for reversals, not enough trend to override them.

VIX_TICKER = '^VIX'

def categorize_vix(df, column_name='vix_close'):
    """
    Categorizes VIX using fixed research-backed thresholds (absolute, not dataset-relative).

    Unlike open_close_diff which uses dynamic mean/std bins, VIX has well-established
    absolute regime boundaries used across quantitative finance literature.

    Categories:
      'Low'      = VIX < 15  (too calm; reversals unreliable, insufficient volatility)
      'Moderate' = 15-25     (sweet spot for mean-reversion)
      'Elevated' = 25-35     (workable but trending behavior increases)
      'High'     = > 35      (panic/crisis; trending dominates, mean-reversion breaks down)

    vix_favorable uses an upper bound of 30 (not 35) because trending/panic behavior
    starts to dominate above 30 even within the 'Elevated' category band.
    """
    vix_bins   = [0, 15, 25, 35, float('inf')]
    vix_labels = ['Low', 'Moderate', 'Elevated', 'High']
    df['vix_category']  = pd.cut(df[column_name], bins=vix_bins, labels=vix_labels, right=False)
    df['vix_favorable'] = df[column_name].between(15, 30)
    return df

# Defensive: ensure date is datetime regardless of upstream dtype
clean['date'] = pd.to_datetime(clean['date'])
start_date = clean['date'].min().strftime('%Y-%m-%d')
end_date   = (clean['date'].max() + pd.Timedelta(days=1)).strftime('%Y-%m-%d')

# Fetch VIX daily closes scoped to dataset date range
vix_raw = yf.download(VIX_TICKER, start=start_date, end=end_date, progress=False)

if vix_raw.empty:
    raise RuntimeError('yfinance returned no VIX data -- check network or date range.')

# Flatten MultiIndex columns produced by newer yfinance versions
if isinstance(vix_raw.columns, pd.MultiIndex):
    vix_raw.columns = vix_raw.columns.get_level_values(0)

vix = vix_raw[['Close']].reset_index()
vix.columns = ['date', 'vix_close']
vix['date'] = pd.to_datetime(vix['date'])

# Map VIX close onto clean by date (faster than merge for one-value-per-day lookups)
vix_map = vix.set_index('date')['vix_close']
clean['vix_close'] = clean['date'].map(vix_map)

# Forward-fill only if there are gaps (expected for weekends/holidays)
if clean['vix_close'].isna().any():
    missing = clean['vix_close'].isna().sum()
    print('WARNING:', missing, 'rows missing VIX close (weekend/holiday gaps) -- filling forward.')
    clean['vix_close'] = clean['vix_close'].ffill()

clean = categorize_vix(clean)

print(clean[['date', 'vix_close', 'vix_category', 'vix_favorable']].drop_duplicates('date').head(20))


           date  vix_close vix_category  vix_favorable
0    2024-03-11  15.220000     Moderate           True
47   2024-03-12  13.840000          Low          False
107  2024-03-13  13.750000          Low          False
168  2024-03-14  14.400000          Low          False
229  2024-03-15  14.410000          Low          False
290  2024-03-18  14.330000          Low          False
350  2024-03-19  13.820000          Low          False
411  2024-03-20  13.040000          Low          False
471  2024-03-21  12.920000          Low          False
532  2024-03-22  13.060000          Low          False
592  2024-03-25  13.190000          Low          False
653  2024-03-26  13.240000          Low          False
713  2024-03-27  12.780000          Low          False
774  2024-03-28  13.010000          Low          False
834  2024-04-01  13.650000          Low          False
895  2024-04-02  14.610000          Low          False
956  2024-04-03  14.330000          Low          False
1017 2024-

In [16]:
clean.to_csv('clean.csv', index=False)